In [ ]:
import os
import numpy as np
import pandas as pd

INPUT_PATH = "/data/hgt/ha-2sec-jma-signed-v2.pqt"
OUTPUT_DIR = "/data/hgt/labels_dp_v2"

# column mapping: set to actual parquet names
COL_ORD     = "bar"
COL_DATE    = "date"
COL_BID     = "bid"      # BBO bid at bar open
COL_ASK     = "ask"      # BBO ask at bar open
COL_HAOPEN  = "haOpen"
COL_HACLOSE = "haClose"
COL_HAHIGH  = "haHigh"
COL_HALOW   = "haLow"
COL_JMA     = "jma"

C_MODE = "colour"                      # "colour" | "midcross"
K_GRID = [0.0, 0.25, 0.5, 1.0, 2.0]    # cost per leg, price units

# ---------------------------------------------------------------------------
# Rule set (Vlad):
# R1 enter long  only if C_long(decision bar) true
# R2 enter short only if C_short(decision bar) true
# R3 exit allowed any bar while in position, but NOT on the entry-fill bar
#    (minimum exposure = 2 bars)
# R4 hold allowed any bar while in position
# R5 stay-away allowed any bar while flat
# R6 no enter decision on the exit-fill bar (one orphan flat bar between trades)
# R7 decision at bar t fills at bar t+1: buy at ask[t+1], sell at bid[t+1];
#    the position column changes on the fill bar, not the decision bar
# R8 flat bars failing C are mechanically stay-away (no decision exists there)
# R9 day ends flat: exit is forced early enough that pos = 0 on the last bar
#
# States (value of pos during the bar):
#   F  = 0  flat, enterable            (R1/R2/R5/R8 apply here)
#   FX = 1  flat, exit filled this bar (R6: enter banned, only stay -> F)
#   L  = 2  long, active               (R3 exit allowed, R4 hold)
#   LF = 3  long, entry filled this bar (R3: exit banned, only hold -> L)
#   S  = 4  short, active
#   SF = 5  short, entry filled this bar
#
# Transitions, decision at t -> state at t+1, reward on the fill (R7):
#   F  -> F   stay                    0
#   F  -> LF  enter long  (R1)       -ask[t+1] - k
#   F  -> SF  enter short (R2)       +bid[t+1] - k
#   FX -> F   stay                    0
#   LF -> L   hold                    0
#   L  -> L   hold                    0
#   L  -> FX  exit long              +bid[t+1] - k
#   SF -> S   hold                    0
#   S  -> S   hold                    0
#   S  -> FX  exit short             -ask[t+1] - k
# No L<->S transition exists (no flip without the flat orphan bar).
# ---------------------------------------------------------------------------

F, FX, L, LF, S, SF = 0, 1, 2, 3, 4, 5
POS_OF = np.array([0, 0, 1, 1, -1, -1], dtype=np.int8)
NEG = -1e18


def build_masks(haOpen, haClose, haHigh, haLow, jma, mode):
    n = haOpen.shape[0]
    if mode == "colour":
        canL = haClose > haOpen          # R1: decision bar is green
        canS = haClose < haOpen          # R2: decision bar is red
    elif mode == "midcross":
        m = (haHigh + haLow) * 0.5
        canL = np.zeros(n, dtype=bool)
        canS = np.zeros(n, dtype=bool)
        canL[1:] = (m[:-1] >= jma[:-1]) & (m[1:] < jma[1:])   # R1: mid crossed JMA down
        canS[1:] = (m[:-1] <= jma[:-1]) & (m[1:] > jma[1:])   # R2: mid crossed JMA up
    else:
        raise ValueError(mode)
    return canL, canS


def label_day(bid, ask, canL, canS, k):
    n = bid.shape[0]
    V = np.full((n, 6), NEG, dtype=np.float64)
    B = np.zeros((n, 6), dtype=np.int8)
    V[0, F] = 0.0                        # day starts flat, enterable

    for t in range(n - 1):
        a1 = ask[t + 1]
        b1 = bid[t + 1]
        vF = V[t, F]; vFX = V[t, FX]; vL = V[t, L]; vLF = V[t, LF]
        vS = V[t, S]; vSF = V[t, SF]

        # -> F : stay from F (R5) or from FX (R6 release)
        if vF >= vFX:
            V[t + 1, F] = vF;  B[t + 1, F] = F
        else:
            V[t + 1, F] = vFX; B[t + 1, F] = FX

        # -> LF : enter long from F (R1, R7)
        if canL[t] and vF > NEG:
            V[t + 1, LF] = vF - a1 - k; B[t + 1, LF] = F

        # -> SF : enter short from F (R2, R7)
        if canS[t] and vF > NEG:
            V[t + 1, SF] = vF + b1 - k; B[t + 1, SF] = F

        # -> L : hold from L (R4) or from LF (R3 release)
        if vL >= vLF:
            V[t + 1, L] = vL;  B[t + 1, L] = L
        else:
            V[t + 1, L] = vLF; B[t + 1, L] = LF

        # -> S : hold from S (R4) or from SF (R3 release)
        if vS >= vSF:
            V[t + 1, S] = vS;  B[t + 1, S] = S
        else:
            V[t + 1, S] = vSF; B[t + 1, S] = SF

        # -> FX : exit long (sell at bid) or exit short (buy at ask) (R3, R7)
        eL = vL + b1 - k if vL > NEG else NEG
        eS = vS - a1 - k if vS > NEG else NEG
        if eL >= eS:
            V[t + 1, FX] = eL; B[t + 1, FX] = L
        else:
            V[t + 1, FX] = eS; B[t + 1, FX] = S

    # R9: terminal must be flat
    s = F if V[n - 1, F] >= V[n - 1, FX] else FX
    total = V[n - 1, s]

    pos = np.empty(n, dtype=np.int8)
    for t in range(n - 1, 0, -1):
        pos[t] = POS_OF[s]
        s = B[t, s]
    pos[0] = POS_OF[s]
    return pos, total


def emit_segments(rows, pos, bid, ask, date, base, gseg, k):
    n = pos.shape[0]
    start = 0
    for i in range(1, n + 1):
        if i < n and pos[i] == pos[i - 1]:
            continue
        end = i - 1
        lab = int(pos[start])
        if lab == 0:
            rows.append((date, gseg, lab, base + start, base + end,
                         end - start + 1, -1, -1, np.nan, np.nan, 0.0))
        else:
            efr = start                          # entry fill bar (R7)
            xfr = end + 1                        # exit fill bar = pos returns to 0
            ep = ask[efr] if lab == 1 else bid[efr]
            xp = bid[xfr] if lab == 1 else ask[xfr]
            util = lab * (xp - ep) - 2.0 * k
            rows.append((date, gseg, lab, base + start, base + end,
                         end - start + 1, base + efr, base + xfr,
                         float(ep), float(xp), float(util)))
        gseg += 1
        start = i
    return gseg


def run():
    cols = [COL_ORD, COL_DATE, COL_BID, COL_ASK,
            COL_HAOPEN, COL_HACLOSE, COL_HAHIGH, COL_HALOW, COL_JMA]
    df = pd.read_parquet(INPUT_PATH, columns=cols)
    df = df.rename(columns={COL_ORD: "ord", COL_DATE: "date", COL_BID: "bid",
                            COL_ASK: "ask", COL_HAOPEN: "haOpen",
                            COL_HACLOSE: "haClose", COL_HAHIGH: "haHigh",
                            COL_HALOW: "haLow", COL_JMA: "jma"})
    df = df.sort_values(["date", "ord"], kind="stable").reset_index(drop=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    dates  = df["date"].to_numpy()
    ord_   = df["ord"].to_numpy()
    bid    = df["bid"].to_numpy(np.float64)
    ask    = df["ask"].to_numpy(np.float64)
    haO    = df["haOpen"].to_numpy(np.float64)
    haC    = df["haClose"].to_numpy(np.float64)
    haH    = df["haHigh"].to_numpy(np.float64)
    haL    = df["haLow"].to_numpy(np.float64)
    jma    = df["jma"].to_numpy(np.float64)

    idx = np.flatnonzero(np.r_[True, dates[1:] != dates[:-1]])
    bounds = np.r_[idx, len(dates)]

    for k in K_GRID:
        all_pos = np.empty(len(dates), dtype=np.int8)
        all_seg = np.empty(len(dates), dtype=np.int32)
        segrows = []
        gseg = 0
        day_utils = []
        for j in range(len(idx)):
            a = bounds[j]; b = bounds[j + 1]
            canL, canS = build_masks(haO[a:b], haC[a:b], haH[a:b], haL[a:b],
                                     jma[a:b], C_MODE)
            pos, total = label_day(bid[a:b], ask[a:b], canL, canS, k)
            all_pos[a:b] = pos
            g0 = gseg
            gseg = emit_segments(segrows, pos, bid[a:b], ask[a:b],
                                 dates[a], a, gseg, k)
            sid = np.empty(b - a, dtype=np.int32)
            start = 0; sc = g0
            for i in range(1, b - a):
                if pos[i] != pos[i - 1]:
                    sid[start:i] = sc; sc += 1; start = i
            sid[start:] = sc
            all_seg[a:b] = sid
            day_utils.append((dates[a], total))

        pd.DataFrame({"ord": ord_, "date": dates, "pos": all_pos,
                      "seg_id": all_seg}).to_parquet(
            os.path.join(OUTPUT_DIR, f"labels_{C_MODE}_k{k:g}.pqt"), index=False)
        pd.DataFrame(segrows, columns=["date", "seg_id", "label", "start_row",
                     "end_row", "n_bars", "entry_fill_row", "exit_fill_row",
                     "entry_price", "exit_price", "utility"]).to_parquet(
            os.path.join(OUTPUT_DIR, f"segments_{C_MODE}_k{k:g}.pqt"), index=False)
        pd.DataFrame(day_utils, columns=["date", "day_utility"]).to_parquet(
            os.path.join(OUTPUT_DIR, f"days_{C_MODE}_k{k:g}.pqt"), index=False)


if __name__ == "__main__":
    run()